In [1]:
import sys
sys.path.append("/host/d/Github/")
import os
import numpy as np
import pandas as pd
import nibabel as nb
import re
import json
import shutil
import Osteosarcoma.functions_collection as ff 
import Osteosarcoma.Data_processing as Data_processing

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Patient split
already done, saved in labels_with_image_info_included_5fold

### Prepare data for nnunet_raw

In [6]:
patient_list = pd.read_excel('/host/e/D/Data/Habitats/External/Patient_lists/image_info_set1.xlsx')
print(patient_list.shape)


save_folder = '/host/e/D/Data/Habitats/External/nnUNet_raw/Dataset603_TumorExternal'
ff.make_folder([save_folder, os.path.join(save_folder, 'imagesTr'), os.path.join(save_folder, 'imagesTs'), os.path.join(save_folder, 'labelsTr')])

for i in range(0, len(patient_list)):
    patient_set = patient_list.iloc[i]['Patient_set']
    patient_index = patient_list.iloc[i]['Patient_index']

    phase = 'Ts'

    # find the image data
    img_file = os.path.join('/host/e/D/Data/Habitats/External/original_data',str(patient_set),str(patient_index),'img.nii.gz')

    # copy image data
    img_save_path = os.path.join(save_folder, 'images' + phase, 'TumorExternal_' + str(patient_set)+ '_' + str(patient_index).zfill(4) + '_0000.nii.gz')
    shutil.copyfile(img_file, img_save_path)

    # find the mask data
    if phase == 'Tr':
        mask_file = os.path.join('/host/e/D/Data/Habitats/External/original_data',str(patient_set),str(patient_index),'label.nii.gz')

        mask_save_path = os.path.join(save_folder, 'labels' + phase, 'TumorExternal_' + str(patient_set)+ '_' + str(patient_index).zfill(4) + '.nii.gz')
        shutil.copyfile(mask_file, mask_save_path)


# borrow Jishuitan dataset for training
patient_list = pd.read_excel('/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_info_set12.xlsx')
for i in range(0,1):
    patient_set = patient_list.iloc[i]['Patient_set']
    patient_index = patient_list.iloc[i]['Patient_index']

    phase = 'Tr'

    # find the image data
    img_file = os.path.join('/host/e/D/Data/Habitats/Jishuitan/original_data',str(patient_set),str(patient_index),'img.nii.gz')

    # copy image data
    img_save_path = os.path.join(save_folder, 'images' + phase, 'TumorInternal_' + str(patient_set)+ '_' + str(patient_index).zfill(4) + '_0000.nii.gz')
    shutil.copyfile(img_file, img_save_path)

    # find the mask data
    if phase == 'Tr':
        mask_file = os.path.join('/host/e/D/Data/Habitats/Jishuitan/original_data',str(patient_set),str(patient_index),'label.nii.gz')

        mask_save_path = os.path.join(save_folder, 'labels' + phase, 'TumorInternal_' + str(patient_set)+ '_' + str(patient_index).zfill(4) + '.nii.gz')
        shutil.copyfile(mask_file, mask_save_path)  

(48, 10)


In [4]:
patient_list_set1 = pd.read_excel('/host/d/Data/Habitats/Jishuitan/Patient_lists/labels_with_image_info_included_set1.xlsx')
patient_set_list_set1 = patient_list_set1['Patient_set']
patient_index_list_set1 = patient_list_set1['Patient_index']
phase_list_set1 = ['Tr'] * len(patient_list_set1)
patient_list_set2 = pd.read_excel('/host/d/Data/Habitats/Jishuitan/Patient_lists/image_info_set2.xlsx')
patient_set_list_set2 = patient_list_set2['Patient_set']
patient_index_list_set2 = patient_list_set2['Patient_index']
phase_list_set2 = ['Tr' if patient_list_set2.iloc[i]['Have_seg'] == 'Yes' else 'Ts' for i in range(len(patient_list_set2))]


# concatenate 
patient_set_list = pd.concat([patient_set_list_set1, patient_set_list_set2], ignore_index=True)
patient_index_list = pd.concat([patient_index_list_set1, patient_index_list_set2], ignore_index=True)
phase_list = pd.concat([pd.Series(phase_list_set1), pd.Series(phase_list_set2)], ignore_index=True)


save_folder = '/host/d/Data/Habitats/Jishuitan/nnUNet_raw/Dataset602_Tumor'
ff.make_folder([save_folder, os.path.join(save_folder, 'imagesTr'), os.path.join(save_folder, 'imagesTs'), os.path.join(save_folder, 'labelsTr')])

for i in range(0, len(patient_set_list)):
    patient_set = patient_set_list.iloc[i]
    patient_index = patient_index_list.iloc[i]
    phase = phase_list.iloc[i]


    # find the image data
    img_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data',patient_set, str(patient_index),'img.nii.gz')

    # copy image data
    img_save_path = os.path.join(save_folder, 'images' + phase, 'Tumor_'+patient_set +'_' + str(patient_index).zfill(4) + '_0000.nii.gz')
    if os.path.isfile(img_save_path) == False:
        shutil.copyfile(img_file, img_save_path)

    # find the mask data
    if phase == 'Tr':
        mask_file = os.path.join('/host/d/Data/Habitats/Jishuitan/original_data',patient_set, str(patient_index),'label.nii.gz')

        mask_save_path = os.path.join(save_folder, 'labels' + phase, 'Tumor_'+patient_set +'_' + str(patient_index).zfill(4) + '.nii.gz')
        shutil.copyfile(mask_file, mask_save_path)


### write the json file

In [8]:
# write the json file
save_folder = '/host/e/D/Data/Habitats/External/nnUNet_raw/Dataset603_TumorExternal'
json_example = os.path.join(save_folder, 'dataset_raw.json')
with open(json_example, 'r') as file:
    data = json.load(file)

# Now 'data' is a Python dictionary or list containing the JSON data
print(data)

{'channel_names': {'0': 'MR image'}, 'labels': {'background': 0, 'tumor': 1}, 'numTraining': 19, 'file_ending': '.nii.gz'}


In [9]:


train_list = []
test_list = []

patient_list = pd.read_excel('/host/e/D/Data/Habitats/External/Patient_lists/image_info_set1.xlsx')
for i in range(0,len(patient_list)):
    patient_set = patient_list.iloc[i]['Patient_set']
    patient_index = patient_list.iloc[i]['Patient_index']
    patient_name = 'TumorExternal_' + str(patient_set)+ '_' + str(patient_index).zfill(4)
    phase = 'Ts'

    if phase == 'Tr':
        train_list.append({'image': "./images%s/%s_0000.nii.gz" % (phase, patient_name), 'label': "./labels%s/%s.nii.gz" % (phase, patient_name)})
    else:
        phase = 'Ts'
        test_list.append("./images%s/%s_0000.nii.gz" % (phase, patient_name))

patient_list = pd.read_excel('/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_info_set12.xlsx')
for i in range(0, 1):
    patient_set = patient_list.iloc[i]['Patient_set']
    patient_index = patient_list.iloc[i]['Patient_index']
    patient_name = 'TumorInternal_' + str(patient_set)+ '_' + str(patient_index).zfill(4)
    phase = 'Tr'

    if phase == 'Tr':
        train_list.append({'image': "./images%s/%s_0000.nii.gz" % (phase, patient_name), 'label': "./labels%s/%s.nii.gz" % (phase, patient_name)})

data["training"] = train_list
data["test"] = test_list
data["numTraining"] = len(train_list)
data["numTest"] = len(test_list)

save_json_file = os.path.join(save_folder, 'dataset.json')
with open(save_json_file, 'w') as file:
    json.dump(data, file, indent=4)